## Dataset download to train 1) MLP, and 2) LSTM with MLP already trained

In [ ]:
! pwd

/content


### Imports

In [ ]:
from google.colab import drive # mount drive for easier export
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# import modules

import os
import shutil
import numpy as np
import pandas as pd

import torch

import kagglehub

from torch.utils.data import TensorDataset
from sklearn.preprocessing import LabelEncoder

import gc
from glob import glob
from google.colab import files as colab_files

import pyarrow as pa
import pyarrow.parquet as pq

import json

In [ ]:
# check if GPU available
gpu_av=torch.cuda.is_available()

# for reproducibility
SEED = 42
torch.manual_seed(SEED)
print("GPU available:", gpu_av)

if gpu_av:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    #print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

print(f"CPU cores: {os.cpu_count()}")


GPU available: False
CPU cores: 2


### Background - already existing functions

In [ ]:
def download_dataset(
    year_start,
    year_end_exd,  # end year (excluded)
    origin_path="flnny123/mfddmulti-modal-flight-delay-dataset/versions/4",
    mode="tabular",  # or "sequential" for pre-made chains
    output_dir_seq="data/chain/",
):

    dest_paths = []
    for year in range(year_start, year_end_exd):
        print(f"Downloading year {year} data...")
        if mode == "tabular":  # tabular dataset download
            origin_path_year = (
                "Aeolus/Flight_Tab/flight_with_weather_" + str(year) + ".csv"
            )
            dest_path_year = kagglehub.dataset_download(
                origin_path, path=origin_path_year
            )
            dest_paths.append(dest_path_year)

        # dest_path_year = dest_path+'flight_with_weather_'+str(year)+'.csv' # destination path
        elif mode == "sequential":  # sequential (chains) dataset download
            for split in ["train", "val", "test"]:
                if year == 2024:
                    origin_path_year_split = f"Aeolus/Flight_chain/chain_data_{year}/flight_chain_{split}_{year}.pt"
                # different naming convention in original dataset
                else:
                    origin_path_year_split = f"Aeolus/Flight_chain/chain_data_{year}/{split}_flight_chain_{year}.pt"

                final_path = os.path.join(
                    output_dir_seq + str(year), f"{split}_flight_chain_{year}.pt"
                )
                if os.path.exists(final_path):
                    print(f"Path {final_path} already exists! Skipping it")
                    continue

                dest_path_year = kagglehub.dataset_download(
                    origin_path, path=origin_path_year_split
                )
                os.makedirs(output_dir_seq + str(year), exist_ok=True)
                shutil.move(dest_path_year, final_path)
                dest_path_year = final_path
                dest_paths.append(final_path)

                print(f"(File(s) available at {dest_path_year}).")



    return dest_paths

In [ ]:
def clean_dataframe(
    df,
    int_type="int32",
    float_type="float32",
    cache_bool=False,
):  # cache=False to save memory at the cost of speed
    # type conversions
    for col in DATE_COLS + DATETIME_COLS:
        df[col] = pd.to_datetime(df[col], format="%Y-%m-%d %H:%M:%S", cache=cache_bool)
    for col in DATE_COLS:
        df[col] = df[col].dt.normalize()  # keep date only, no time (cleaner)
    for col in TIMEDELTA_MINS_COLS:
        df[col] = pd.to_timedelta(df[col], unit="m")
    df[INT_COLS] = df[INT_COLS].astype(int_type)
    df[STR_COLS] = df[STR_COLS].astype("str")
    df[FLOAT_COLS] = df[FLOAT_COLS].astype(float_type)  # limit precision??

    # drop all rows containing NaNs and keep track of them
    n_rows_before = len(df)
    df.dropna(subset=INT_COLS + FLOAT_COLS, inplace=True)
    n_rows_after = len(df)
    n_rows_dropped = n_rows_before - n_rows_after
    print(f"Dropped {n_rows_dropped} rows because of NaNs.")
    nan_bool = bool(np.any(df.isna().sum() > 0))
    print(f"Any NaNs remaining in numerical data: {nan_bool}.")

    return df

In [ ]:
def add_all_features(df: pd.DataFrame, delay_threshold: int = 15) -> pd.DataFrame:
    """Versione memory-safe: una sola copia iniziale, poi tutte le feature
    vengono aggiunte in-place sullo stesso oggetto, invece di incatenare
    funzioni che fanno ciascuna un .copy() completo del dataframe."""
    df = df.copy()

    minutes_in_day = 24 * 60
    df["dep_hour_sin"] = np.sin(2 * np.pi * df["CRS_DEP_TIME_MIN"] / minutes_in_day)
    df["dep_hour_cos"] = np.cos(2 * np.pi * df["CRS_DEP_TIME_MIN"] / minutes_in_day)
    df["arr_hour_sin"] = np.sin(2 * np.pi * df["CRS_ARR_TIME_MIN"] / minutes_in_day)
    df["arr_hour_cos"] = np.cos(2 * np.pi * df["CRS_ARR_TIME_MIN"] / minutes_in_day)

    date = pd.to_datetime(dict(year=df["FL_YEAR"], month=df["FL_MONTH"], day=df["FL_DAY"]))
    dow = date.dt.dayofweek
    df["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    df["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["FL_MONTH"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["FL_MONTH"] / 12)
    del date, dow
    R = 6371.0
    lat1, lon1 = np.radians(df["O_LATITUDE"]), np.radians(df["O_LONGITUDE"])
    lat2, lon2 = np.radians(df["D_LATITUDE"]), np.radians(df["D_LONGITUDE"])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    df["great_circle_km"] = R * 2 * np.arcsin(np.sqrt(a))
    del lat1, lon1, lat2, lon2, a

    df["dep_hour_bucket"] = (df["CRS_DEP_TIME_MIN"] // 120).astype("int8")
    congestion = (
        df.groupby(["ORIGIN_INDEX", "FL_YEAR", "FL_MONTH", "FL_DAY", "dep_hour_bucket"], observed=True)
        .size().rename("origin_congestion_2h").reset_index()
    )
    df = df.merge(congestion, on=["ORIGIN_INDEX", "FL_YEAR", "FL_MONTH", "FL_DAY", "dep_hour_bucket"], how="left")
    del congestion

    df["ARR_DELAY_BIN"] = (df["ARR_DELAY"] > delay_threshold).astype("int8")
    df["DEP_DELAY_BIN"] = (df["DEP_DELAY"] > delay_threshold).astype("int8")

    return df


### New functions

In [ ]:
# new tensor's feature in order: FL_YEAR, FL_DAY, 'CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', o lat, o long, d lat, d long, FL_WEEK, the 10 engineered features

In [ ]:
# features used in MLP:

# 'FL_DAY',  # day of the month - missing

# 'FL_WEEK', # week of the year - missing


# 'dep_hour_sin', 'dep_hour_cos', 'arr_hour_sin', 'arr_hour_cos', 'dow_sin', # calc from others
# 'dow_cos', 'month_sin', 'month_cos', 'great_circle_km', 'origin_congestion_2h',
# 'CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', # can be converted
# # 'CRS_ELAPSED_TIME', 'FLIGHTS', can be deduced
# #  'O_LATITUDE', 'O_LONGITUDE', 'D_LATITUDE', 'D_LONGITUDE',  # can be deduced
#  'FL_YEAR', # can be deduced
# 'FL_MONTH',
# 'ORIGIN_INDEX', 'DEST_INDEX',
# 'O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD',
# 'OP_CARRIER', 'OP_CARRIER_FL_NUM',

In [ ]:
# features used in LSTM chains:
# 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR',

# 'MONTH',
# 'ORIGIN_INDEX', 'DEST_INDEX',
# 'OP_CARRIER', 'OP_CARRIER_FL_NUM'
# 'O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD'

In [ ]:
MAX_SEQ_LEN = 6

DENSE_FEAT_COLS = ["O_TEMP", "D_TEMP", "O_PRCP", "D_PRCP", "O_WSPD", "D_WSPD"]
SPARSE_FEAT_COLS = ["MONTH_0", "DOW_0", "CRS_ARR_TIME_HOUR", "CRS_DEP_TIME_HOUR",
                     "ORIGIN_INDEX", "DEST_INDEX", "OP_CARRIER", "OP_CARRIER_FL_NUM"]
TARGET_COLS = ["ARR_DELAY", "DEP_DELAY"]        # numeric minutes -> "delays" tensor
LABEL_COLS = ["ARR_DELAY_BIN", "DEP_DELAY_BIN"]  # -> "labels" tensor

# IMPORTANT NOTE: THIS IS THE ORDER of additional features
MLP_ONLY_COLS = [
    "FL_YEAR", "FL_DAY", "FL_WEEK",
    "CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN",
    "O_LATITUDE", "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE",
    "dep_hour_sin", "dep_hour_cos", "arr_hour_sin", "arr_hour_cos",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "great_circle_km", "origin_congestion_2h",
]  # order matters -- treat as metadata for the 6th tensor's columns

def merge_and_save_split(split,year_str):
    file_list = sorted(glob(f"tmp_{split}_*.pt"))
    if not file_list:
        print(f"No files for split {split}")
        return None
    merged = None
    for f in file_list:
        ds = torch.load(f, weights_only=False)
        if merged is None:
            merged = ds
        else:
            # Concatenate all tensors
            merged = TensorDataset(
                torch.cat([merged.tensors[0], ds.tensors[0]], dim=0),
                torch.cat([merged.tensors[1], ds.tensors[1]], dim=0),
                torch.cat([merged.tensors[2], ds.tensors[2]], dim=0),
                torch.cat([merged.tensors[3], ds.tensors[3]], dim=0),
                torch.cat([merged.tensors[4], ds.tensors[4]], dim=0),
                torch.cat([merged.tensors[5], ds.tensors[5]], dim=0),
            )
        os.remove(f)
     # Save merged dataset to a file
    out_file = f"new_{split}_flight_chain_{year_str}.pt"
    torch.save(merged, out_file)
    print(f"Saved {out_file}.")

    return "Success."


def compute_day_splits(df, seed=42, train_frac=0.6, valid_frac=0.2):
    """Assigns each unique flight DAY to train/val/test (mirrors prepare_data())."""
    date_dim = df[["FL_YEAR", "FL_MONTH", "FL_DAY"]].drop_duplicates().reset_index(drop=True)
    date_dim["DAY_OF_YEAR"] = pd.to_datetime(
        dict(year=date_dim["FL_YEAR"], month=date_dim["FL_MONTH"], day=date_dim["FL_DAY"])
    ).dt.dayofyear

    rng = np.random.RandomState(seed=seed)
    train_days, valid_days, test_days = [], [], []

    for month in sorted(date_dim["FL_MONTH"].unique()):
        month_dates = date_dim[date_dim["FL_MONTH"] == month]["DAY_OF_YEAR"]
        n_days = len(month_dates)
        if n_days < 3:
            raise Exception(f"Fewer than 3 days for month {month} -- can't guarantee a 3-way split.")
        mandatory_days = rng.choice(month_dates, 3, replace=False)
        train_days.append(mandatory_days[0])
        valid_days.append(mandatory_days[1])
        test_days.append(mandatory_days[2])

        remaining_days = [d for d in month_dates if d not in mandatory_days]
        n_remaining = len(remaining_days)
        if n_remaining > 0:
            permuted = rng.permutation(remaining_days)
            split1 = int(round(n_remaining * train_frac))
            split2 = split1 + int(round(n_remaining * valid_frac))
            train_days.extend(permuted[:split1])
            valid_days.extend(permuted[split1:split2])
            test_days.extend(permuted[split2:])

    date_dim["SPLIT_TYPE"] = np.select(
        [date_dim["DAY_OF_YEAR"].isin(train_days),
         date_dim["DAY_OF_YEAR"].isin(valid_days),
         date_dim["DAY_OF_YEAR"].isin(test_days)],
        ["train", "val", "test"],
        default="undefined",
    )
    return date_dim[["FL_YEAR", "FL_MONTH", "FL_DAY", "SPLIT_TYPE"]]


def split_by_day(df, seed=42, train_frac=0.6, valid_frac=0.2):
    day_splits = compute_day_splits(df, seed=seed, train_frac=train_frac, valid_frac=valid_frac)
    return df.merge(day_splits, on=["FL_YEAR", "FL_MONTH", "FL_DAY"], how="left")


# ---------------------------------------------------------------------------
# STEP "full" mode: build chains -> 6-tensor PyTorch datasets
# ---------------------------------------------------------------------------
def iter_flight_chains(df):
    df_sorted = df.sort_values(by=["OP_CARRIER", "OP_CARRIER_FL_NUM", "FL_DATE", "CRS_DEP_TIME"])
    grouped = df_sorted.groupby(["OP_CARRIER", "OP_CARRIER_FL_NUM", "FL_DATE"])
    return grouped # Returns the GroupBy object, which is an iterator

# Truncation/padding for single chain
def adjust_sequence(data, max_len=MAX_SEQ_LEN):
    if len(data) < max_len:
        pad_shape = (max_len - len(data), data.shape[1])
        return torch.cat([data, torch.zeros(pad_shape, dtype=data.dtype)], dim=0)
    return data[:max_len]


def process_all_chains_full(flight_chains_iterator, max_sequence_length=MAX_SEQ_LEN):
    """First 5 tensors are EXACTLY the existing LSTM format; 6th is new (MLP feats)."""
    processed = []
    for name, chain in flight_chains_iterator:
        dense_feat = torch.stack(
            [torch.tensor(chain[c].fillna(0).values, dtype=torch.float32) for c in DENSE_FEAT_COLS],
            dim=1,
        )
        sparse_feat = torch.stack(
            [torch.tensor(chain[c].values.astype(np.int16), dtype=torch.int16) for c in SPARSE_FEAT_COLS],
            dim=1,
        )
        labels = torch.tensor(chain[LABEL_COLS].values.astype(np.int8), dtype=torch.int8)
        delays = torch.tensor(chain[TARGET_COLS].values.astype(np.int16), dtype=torch.int16)
        mlp_feat = torch.tensor(chain[MLP_ONLY_COLS].fillna(0).values, dtype=torch.float32)

        valid_len = min(len(chain), max_sequence_length)

        dense_feat = adjust_sequence(dense_feat, max_sequence_length)
        sparse_feat = adjust_sequence(sparse_feat, max_sequence_length)
        labels = adjust_sequence(labels, max_sequence_length)
        delays = adjust_sequence(delays, max_sequence_length)
        mlp_feat = adjust_sequence(mlp_feat, max_sequence_length)

        processed.append((dense_feat, sparse_feat, labels, valid_len, delays, mlp_feat))
    return processed


def create_dataset_full(processed_data):
    dense = torch.stack([item[0] for item in processed_data])
    sparse = torch.stack([item[1] for item in processed_data])
    labels = torch.stack([item[2] for item in processed_data])
    valid_lens = torch.tensor([item[3] for item in processed_data], dtype=torch.long)
    delays = torch.stack([item[4] for item in processed_data])
    mlp_feat = torch.stack([item[5] for item in processed_data])
    return TensorDataset(dense, sparse, labels, valid_lens, delays, mlp_feat)


# ---------------------------------------------------------------------------
# STEP 2b -- "mlp" mode: flight-level parquet export (not used)
# ---------------------------------------------------------------------------
MLP_EXPORT_EXTRA_EXCLUDE = {
    # raw datetime/timedelta/string columns superseded by derived features
    "CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME", "ARR_TIME", "WHEELS_OFF", "WHEELS_ON",
    "TAXI_OUT", "TAXI_IN", "CRS_ELAPSED_TIME", "ACTUAL_ELAPSED_TIME", "AIR_TIME",
    "ORIGIN", "DEST",
    # raw 1-indexed / LSTM-only helper columns, superseded by FL_MONTH/FL_DAY/FL_WEEK
    "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "MONTH_0", "DOW_0",
    # bookkeeping, not a feature
    "SPLIT_TYPE",
}


def export_mlp_parquets(df, output_dir="mlp_data"):
    exclude_cols = {"ARR_DELAY", "DEP_DELAY", "ARR_DELAY_BIN", "DEP_DELAY_BIN",
                     "FL_DATE", "dep_hour_bucket"} | MLP_EXPORT_EXTRA_EXCLUDE
    target_cols = ["ARR_DELAY_BIN", "DEP_DELAY_BIN"]
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    os.makedirs(output_dir, exist_ok=True)
    paths = {}
    for split in ["train", "val", "test"]:
        split_df = df.loc[df["SPLIT_TYPE"] == split, feature_cols + target_cols]
        out_path = os.path.join(output_dir, f"mlp_{split}.parquet")
        split_df.to_parquet(out_path, index=False)
        paths[split] = out_path
        print(f"Saved {split}: {len(split_df)} rows -> {out_path}")
        del split_df # Free up memory
        gc.collect()
    return paths


# ---------------------------------------------------------------------------
# TOP-LEVEL ENTRY POINT
# ---------------------------------------------------------------------------
def prepare_multi_head_dataset(
    df, splits, mode="full", # or "mlp" (not used)
    seed=42, train_frac=0.6, valid_frac=0.2,
    max_sequence_length=MAX_SEQ_LEN, output_dir="prepared_data",
):
    print("Starting prepare_multi_head_dataset...")
    df = split_by_day(df, seed=seed, train_frac=train_frac, valid_frac=valid_frac)
    print("Finished split_by_day.")

    if mode == "mlp":
        return export_mlp_parquets(df, output_dir=output_dir)

    elif mode == "full":
        datasets = {}
        for split in splits: #["train", "val", "test"]:
            split_df = df[df["SPLIT_TYPE"] == split]
            print(f"Processing split: {split} with {len(split_df)} rows...")
            all_chains = []
            for date, group in split_df.groupby("FL_DATE"):
                # group is a DataFrame for one day – fits in memory
                chains = iter_flight_chains(group)
                processed = process_all_chains_full(chains, max_sequence_length=max_sequence_length)
                day_dataset = create_dataset_full(processed)
                # Instead of accumulating all in RAM, write to disk
                date_str = date.strftime('%Y_%m_%d')
                out_file = f"tmp_{split}_{date_str}.pt"
                torch.save(day_dataset, out_file)
                print(f"Saved {out_file} with {len(day_dataset)} chains")
                del group, chains, processed, day_dataset
                gc.collect()
            del split_df
            gc.collect()
        return f"Success! Now, manually download {split}"

    else:
        raise ValueError(f"Unknown mode: {mode!r}")


In [ ]:
def prepare_dataframe_common_chunk_safe(df):
    """Chunk-safe subset: stateless cleaning + feature engineering only.
    NO LabelEncoder fitting here -- that must happen on the full dataset."""
    df = clean_dataframe(df)

    df["FL_YEAR"] = df["FL_DATE"].dt.year.astype("int16")
    df["FL_MONTH"] = df["FL_DATE"].dt.month.astype("int8")
    df["FL_DAY"] = df["FL_DATE"].dt.day.astype("int8")
    df["FL_WEEK"] = df["FL_DATE"].dt.isocalendar().week.astype("int8")

    df["CRS_DEP_TIME_MIN"] = (df["CRS_DEP_TIME"].dt.hour * 60 + df["CRS_DEP_TIME"].dt.minute).astype("int16")
    df["CRS_ARR_TIME_MIN"] = (df["CRS_ARR_TIME"].dt.hour * 60 + df["CRS_ARR_TIME"].dt.minute).astype("int16")
    df["CRS_DEP_TIME_HOUR"] = df["CRS_DEP_TIME"].dt.hour.astype("int8")
    df["CRS_ARR_TIME_HOUR"] = df["CRS_ARR_TIME"].dt.hour.astype("int8")

    df["ARR_DELAY"] = (df["ARR_DELAY"].dt.total_seconds() / 60).astype("int16")
    df["DEP_DELAY"] = (df["DEP_DELAY"].dt.total_seconds() / 60).astype("int16")

    df["MONTH_0"] = (df["MONTH"] - 1).astype("int16")
    df["DOW_0"] = (df["DAY_OF_WEEK"] - 1).astype("int16")

    df = add_all_features(df, delay_threshold=15)  # no fitting inside, safe
    return df


def convert_csv_to_parquet(csv_path, parquet_path, chunksize=100_000):
    writer = None
    for chunk in pd.read_csv(csv_path, chunksize=chunksize, low_memory=True, usecols=usecols, dtype=DTYPE_MAPPING):
        processed = prepare_dataframe_common_chunk_safe(chunk)
        table = pa.Table.from_pandas(processed, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(parquet_path, table.schema)
        writer.write_table(table)
    if writer is not None:
        writer.close()
    print(f"Parquet saved to {parquet_path}")


In [ ]:
def build_and_save_category_mapping(categories, save_path):
    """categories: any iterable of unique values (e.g. from one year, or a
    known reference list of all carriers/flight numbers)."""
    unique_sorted = sorted(set(categories))
    mapping = {val: i for i, val in enumerate(unique_sorted)}
    with open(save_path, "w") as f:
        json.dump({str(k): v for k, v in mapping.items()}, f)
    return mapping

def load_category_mapping(save_path):
    with open(save_path) as f:
        raw = json.load(f)
    return raw

def apply_category_mapping(series, mapping, unknown_value=-1):
    """Unseen categories (e.g. a new carrier code in 2023 not present in 2022)
    map to unknown_value -- flag this if it happens, don't silently ignore it."""
    return series.astype(str).map(mapping).fillna(unknown_value).astype("int16")

### EXAMPLE USE
The following procedure was used to create the new, "merged" datasets for LSTM+MLP model

In [ ]:
# define column names of original tabular dataset as global variables
# (for dataset download later)
DATE_COLS=["FL_DATE"]
DATETIME_COLS=["CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME",
               "ARR_TIME", "WHEELS_OFF", "WHEELS_ON", ]
TIMEDELTA_MINS_COLS=["DEP_DELAY", "ARR_DELAY", "TAXI_OUT", "TAXI_IN", "CRS_ELAPSED_TIME",
                     "ACTUAL_ELAPSED_TIME", "AIR_TIME",	]
INT_COLS=["OP_CARRIER_FL_NUM", "MONTH", "DAY_OF_MONTH",
          "DAY_OF_WEEK", "ORIGIN_INDEX", "DEST_INDEX"] # removed "FLIGHTS" bc constant
STR_COLS=["OP_CARRIER", "ORIGIN", "DEST"]
FLOAT_COLS=["O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD", "O_LATITUDE",
             "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE"]
size_set_check=set(DATE_COLS+DATETIME_COLS+TIMEDELTA_MINS_COLS+INT_COLS+STR_COLS+FLOAT_COLS)
print(f"Total individual features (should be 34): {len(size_set_check)}")


DTYPE_MAPPING = {}
# 1. Date/time and string columns → read as strings (object), convert later
for col in DATE_COLS + DATETIME_COLS + STR_COLS:
    DTYPE_MAPPING[col] = 'object'

# 2. Floating point columns (weather, coordinates) -> float32
for col in FLOAT_COLS:
    DTYPE_MAPPING[col] = 'float32'

# 3. Integer columns – assign individually to save memory
#    - OP_CARRIER_FL_NUM: up to ~10k, int32 for safety
DTYPE_MAPPING['OP_CARRIER_FL_NUM'] = 'int32'
#    - FLIGHTS: always 1 → int8
#DTYPE_MAPPING['FLIGHTS'] = 'int8'
#    - MONTH, DAY_OF_MONTH, DAY_OF_WEEK: ranges 1‑31, 1‑12, 1‑7 -> int8
for col in ['MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK']:
    DTYPE_MAPPING[col] = 'int8'
#    - ORIGIN_INDEX, DEST_INDEX: number of airports (~300) -> int16
for col in ['ORIGIN_INDEX', 'DEST_INDEX']:
    DTYPE_MAPPING[col] = 'int16'

# 4. Delay/time columns (minutes, can be negative) -> int16 (range -32768..32767)
for col in TIMEDELTA_MINS_COLS:
    DTYPE_MAPPING[col] = 'int16'


usecols = DATE_COLS + DATETIME_COLS + TIMEDELTA_MINS_COLS + STR_COLS + FLOAT_COLS + INT_COLS

Total individual features (should be 34): 33


In [ ]:
path_to_2022_csv, path_to_2023_csv = download_dataset(2022, 2024)

In [ ]:
# Convert once to CSV
# NOTE: we drop all rows with NaNs, while original paper filled with zeros; hence different delay rate

convert_csv_to_parquet(path_to_2022_csv, "data_2022.parquet")
convert_csv_to_parquet(path_to_2023_csv, "data_2023.parquet")


Dropped 7 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 19 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 2 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 11 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 4 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 2 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 6 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 9 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 3 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 2 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 2 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 5 rows because of NaNs.
Any NaNs remaining in numerical data: False.
Dropped 1 rows because of NaNs.
Any NaNs remaining in numerical data: Fals

```python
carriers_2022 = pd.read_parquet("data_2022.parquet", columns=["OP_CARRIER"])["OP_CARRIER"]
carriers_2023 = pd.read_parquet("data_2023.parquet", columns=["OP_CARRIER"])["OP_CARRIER"]
carrier_mapping = build_and_save_category_mapping(
    pd.concat([carriers_2022, carriers_2023]), "carrier_mapping.json"
)
del carriers_2022, carriers_2023
gc.collect()
flnums_2022 = pd.read_parquet("data_2022.parquet", columns=["OP_CARRIER_FL_NUM"])["OP_CARRIER_FL_NUM"]
flnums_2023 = pd.read_parquet("data_2023.parquet", columns=["OP_CARRIER_FL_NUM"])["OP_CARRIER_FL_NUM"]
flnum_mapping = build_and_save_category_mapping(
    pd.concat([flnums_2022, flnums_2023]), "flnum_mapping.json"
)
del flnums_2022, flnums_2023
gc.collect()

```

### OPERATIONS ON EACH DATAFRAME (df_2022, then delete, then df_2023)

#### 2022

In [ ]:
#del df_2022
#gc.collect()

0

In [ ]:
df_2022 = pd.read_parquet("data_2022.parquet")


In [ ]:
# Apply full mappings, one year in memory at a time
carrier_mapping = load_category_mapping("carrier_mapping.json")
flnum_mapping = load_category_mapping("flnum_mapping.json")

In [ ]:
carrier_mapping

{'9E': 0,
 'AA': 1,
 'AS': 2,
 'B6': 3,
 'DL': 4,
 'F9': 5,
 'G4': 6,
 'HA': 7,
 'MQ': 8,
 'NK': 9,
 'OH': 10,
 'OO': 11,
 'QX': 12,
 'UA': 13,
 'WN': 14,
 'YV': 15,
 'YX': 16}

In [ ]:
df_2022["OP_CARRIER"] = apply_category_mapping(df_2022["OP_CARRIER"], carrier_mapping)
df_2022["OP_CARRIER_FL_NUM"] = apply_category_mapping(df_2022["OP_CARRIER_FL_NUM"], flnum_mapping)

In [ ]:
df_2022.tail() # sanity check

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,...,arr_hour_cos,dow_sin,dow_cos,month_sin,month_cos,great_circle_km,dep_hour_bucket,origin_congestion_2h,ARR_DELAY_BIN,DEP_DELAY_BIN
6412250,2022-05-27,9,224,MYR,IAG,2022-05-27 16:00:00,2022-05-27 15:52:00,-8,0 days 00:11:00,2022-05-27 16:03:00,...,-0.03926,-0.433884,-0.900969,5.000000e-01,-0.866025,1048.292603,8,1,0,0
6412251,2022-05-28,9,224,MYR,IAG,2022-05-28 16:00:00,2022-05-28 15:44:00,-16,0 days 00:16:00,2022-05-28 16:00:00,...,-0.03926,-0.974928,-0.222521,5.000000e-01,-0.866025,1048.292603,8,1,0,0
6412252,2022-05-30,9,224,MYR,IAG,2022-05-30 16:00:00,2022-05-30 15:51:00,-9,0 days 00:12:00,2022-05-30 16:03:00,...,-0.03926,0.000000,1.000000,5.000000e-01,-0.866025,1048.292603,8,1,0,0
6412253,2022-06-03,9,224,MYR,IAG,2022-06-03 16:00:00,2022-06-03 16:08:00,8,0 days 00:13:00,2022-06-03 16:21:00,...,-0.03926,-0.433884,-0.900969,1.224647e-16,-1.000000,1048.292603,8,1,0,0
6412254,2022-06-04,9,224,MYR,IAG,2022-06-04 16:00:00,2022-06-04 15:48:00,-12,0 days 00:09:00,2022-06-04 15:57:00,...,-0.03926,-0.974928,-0.222521,1.224647e-16,-1.000000,1048.292603,8,1,0,0


##### Procedure to obtain train / val / test data
(here done for test)

In [ ]:
gc.collect()
prepare_multi_head_dataset(df_2022, mode="full", splits=["test"]) # output: full pytorch dataset for MLP+LSTM


Starting prepare_multi_head_dataset...
Finished split_by_day.
Processing split: test with 1332579 rows...
Saved tmp_test_2022_01_10.pt with 13838 chains
Saved tmp_test_2022_01_15.pt with 11717 chains
Saved tmp_test_2022_01_17.pt with 13415 chains
Saved tmp_test_2022_01_20.pt with 13949 chains
Saved tmp_test_2022_01_24.pt with 13935 chains
Saved tmp_test_2022_01_29.pt with 9488 chains
Saved tmp_test_2022_02_14.pt with 14459 chains
Saved tmp_test_2022_02_16.pt with 13626 chains
Saved tmp_test_2022_02_17.pt with 14015 chains
Saved tmp_test_2022_02_19.pt with 12597 chains
Saved tmp_test_2022_02_20.pt with 13781 chains
Saved tmp_test_2022_02_23.pt with 13031 chains
Saved tmp_test_2022_03_04.pt with 14668 chains
Saved tmp_test_2022_03_08.pt with 13516 chains
Saved tmp_test_2022_03_10.pt with 14853 chains
Saved tmp_test_2022_03_16.pt with 14046 chains
Saved tmp_test_2022_03_17.pt with 14450 chains
Saved tmp_test_2022_03_26.pt with 13419 chains
Saved tmp_test_2022_04_02.pt with 12032 chains
Sa

'Success! Now, manually download test'

In [ ]:
gc.collect()
merge_and_save_split("test", "2022")
gc.collect()


Saved new_test_flight_chain_2022.pt.


339

In [ ]:
shutil.copy("new_test_flight_chain_2022.pt", "/content/drive/MyDrive/flight_delay_results/new_test_flight_chain_2022.pt")
print("Copied to Drive.")

Copied to Drive.


In [ ]:
# SANITY CHECK

file_path="/content/drive/MyDrive/flight_delay_results/new_test_flight_chain_2022.pt"

def inspect_pt_dataset(file_path):
    """
    Inspects a PyTorch TensorDataset stored in a .pt file, providing a summary
    of its contents, shapes, data types, and specific characteristics like
    label distribution and unmapped categorical entries. It also performs
    a consistency check for cyclic time features.

    Args:
        file_path (str): The path to the .pt file containing the TensorDataset.
    """

    # Load the TensorDataset from the specified file.
    # weights_only=False ensures that the actual tensor data is loaded,
    # not just metadata if the file were a model checkpoint.
    data = torch.load(file_path, weights_only=False)

    print(f"Type of loaded object: {type(data)}")

    # Check if the loaded object is indeed a TensorDataset by looking for the 'tensors' attribute.
    if hasattr(data, 'tensors'):
        print(f"Number of tensors: {len(data.tensors)}")
        # Iterate through each tensor in the dataset to print its shape and data type.
        for i, tensor in enumerate(data.tensors):
            print(f"Tensor {i}: shape={tensor.shape}, dtype={tensor.dtype}")

        # Tensor 3 contains the valid sequence lengths for each flight chain.
        # This indicates the actual number of flights in a sequence before padding.
        valid_lens = data.tensors[3]
        print(f"\nValid lengths: min={valid_lens.min()}, max={valid_lens.max()}, mean={valid_lens.float().mean():.2f}")

        # Tensor 2 contains the binary labels for flight delays (e.g., ARR_DELAY_BIN, DEP_DELAY_BIN).
        # We analyze the distribution of '1's (delayed flights) for each label column.
        labels = data.tensors[2]
        print(f"\nLabel distribution (0=on-time, 1=delayed):")
        # Assuming there are two label columns (e.g., for arrival and departure delays).
        for col in [0, 1]:
            # Count the number of '1's (delayed) in the current label column across all sequences.
            pos = (labels[:, :, col] == 1).sum().item()
            # Get the total number of elements in that label column (total possible flights).
            total = labels[:, :, col].numel()
            # Print the count and percentage of positive (delayed) labels.
            print(f"  Column {col}: {pos}/{total} ({pos/total*100:.1f}%) positive")

        # Tensor 1 contains sparse (categorical) features, including OP_CARRIER and OP_CARRIER_FL_NUM.
        # These are encoded using LabelEncoder, and -1 indicates an unmapped (unknown) category.
        sparse = data.tensors[1]
        # Check for unmapped carrier entries (index 6 in SPARSE_FEAT_COLS).
        n_unmapped_carrier = (sparse[:, :, 6] == -1).sum().item()
        # Check for unmapped flight number entries (index 7 in SPARSE_FEAT_COLS).
        n_unmapped_flnum = (sparse[:, :, 7] == -1).sum().item()
        print(f"Unmapped carrier entries: {n_unmapped_carrier}")
        print(f"Unmapped flnum entries: {n_unmapped_flnum}")

        # --- Random-sample consistency check for MONTH_0/DOW_0 vs. month_sin/cos ---
        # Tensor 5 contains MLP-only features, which include cyclic time features like month_sin/cos.
        # This section randomly samples a few data points and compares the raw month/day_of_week indices
        # from sparse features with their corresponding sine/cosine transformations in MLP features.
        mlp = data.tensors[5]
        # Select 10 random indices from the dataset for inspection.
        idxs = torch.randint(0, len(data), (10,))
        print("\nRandom sample MONTH_0/DOW_0 vs month_sin/cos check:")
        for i in idxs.tolist(): # Iterate through the randomly selected indices.
            # Extract MONTH_0 (0-indexed month) and DOW_0 (0-indexed day of week) from the sparse features.
            # We take the first entry (index 0) of the sequence for simplicity.
            month0 = sparse[i, 0, 0].item()
            dow0 = sparse[i, 0, 1].item()
            # Extract month_sin and month_cos from the MLP features.
            # These are at specific indices (15 and 16) within the MLP_ONLY_COLS list.
            month_sin, month_cos = mlp[i, 0, 15].item(), mlp[i, 0, 16].item()
            print(f"  idx {i}: MONTH_0={month0}, DOW_0={dow0}, month_sin/cos={month_sin:.2f}/{month_cos:.2f}")

    else:
        # If the loaded object is not a TensorDataset, print its contents directly.
        print("Object is not a TensorDataset. Contents:")
        print(data)

    # Calculate and print the total memory usage of all tensors in the dataset.
    # This helps in understanding the dataset's memory footprint.
    total_bytes = sum(t.element_size() * t.nelement() for t in data.tensors)
    print(f"\nTotal memory: {total_bytes / 1024**2:.2f} MB")

inspect_pt_dataset(file_path)

Type of loaded object: <class 'torch.utils.data.dataset.TensorDataset'>
Number of tensors: 6
Tensor 0: shape=torch.Size([1080843, 6, 6]), dtype=torch.float32
Tensor 1: shape=torch.Size([1080843, 6, 8]), dtype=torch.int16
Tensor 2: shape=torch.Size([1080843, 6, 2]), dtype=torch.int8
Tensor 3: shape=torch.Size([1080843]), dtype=torch.int64
Tensor 4: shape=torch.Size([1080843, 6, 2]), dtype=torch.int16
Tensor 5: shape=torch.Size([1080843, 6, 19]), dtype=torch.float32

Valid lengths: min=1, max=6, mean=1.23

Label distribution (0=on-time, 1=delayed):
  Column 0: 256803/6485058 (4.0%) positive
  Column 1: 261953/6485058 (4.0%) positive
Unmapped carrier entries: 0
Unmapped flnum entries: 0

Random sample MONTH_0/DOW_0 vs month_sin/cos check:
  idx 343158: MONTH_0=4, DOW_0=5, month_sin/cos=0.50/-0.87
  idx 257972: MONTH_0=3, DOW_0=3, month_sin/cos=0.87/-0.50
  idx 942865: MONTH_0=10, DOW_0=3, month_sin/cos=-0.50/0.87
  idx 992710: MONTH_0=10, DOW_0=2, month_sin/cos=-0.50/0.87
  idx 798582: MO

In [ ]:
#********** now download to local !!! ********************

In [ ]:
# Delete all tmp_{split}_*.pt files (per-day intermediate files, in case any remain)
tmp_files = glob("tmp_*.pt")
for f in tmp_files:
    os.remove(f)
print(f"Deleted {len(tmp_files)} tmp files.")

# Delete the final merged/downloaded split file too, since you already
# downloaded it manually (adjust filename pattern as needed)
final_files = glob("new_*_flight_chain_202*.pt")  # or glob("new_*_flight_chain_2022.pt") for all splits/year
for f in final_files:
    os.remove(f)
print(f"Deleted {len(final_files)} final merged files.")


Deleted 0 tmp files.
Deleted 1 final merged files.


In [ ]:
# Delete any leftover intermediate objects from prepare_multi_head_dataset
# (safe even if some of these no longer exist)
for var_name in ["merged", "split_df", "day_dataset", "chains", "processed",
                  "group", "buffer", "merged_buffer"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

10

In [ ]:
# desired output:
# Sample Type: <class 'tuple'>
# Total components in the sample tuple: 5

# ------------------------------------------------------------
# 1. Dense Features (Continuous / Meteorological Features):
#    - Shape: torch.Size([6, 6]) (Sequence Length x 6)
#    - Cols: ['O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD']
#     - Values:
# tensor([[ 6.7000, 19.4000,  0.0000,  0.0000,  9.4000, 14.8000],
#         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
#         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
#         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
#         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
#         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]])

# 2. Sparse Features (Categorical / Temporal Features):
#    - Shape: torch.Size([6, 8]) (Sequence Length x 8)
#    - Cols: ['MONTH', 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR', 'ORIGIN_INDEX', 'DEST_INDEX', 'OP_CARRIER', 'OP_CARRIER_FL_NUM']
#     - Values:
# tensor([[   0,    5,   11,    9,   93,  257,    0, 4623],
#         [   0,    0,    0,    0,    0,    0,    0,    0],
#         [   0,    0,    0,    0,    0,    0,    0,    0],
#         [   0,    0,    0,    0,    0,    0,    0,    0],
#         [   0,    0,    0,    0,    0,    0,    0,    0],
#         [   0,    0,    0,    0,    0,    0,    0,    0]], dtype=torch.int16)

# 3. Binary Labels (Flight Delay Indicators > 15 mins):
#    - Shape: torch.Size([6, 2]) (Sequence Length x 2)
#    - [(ARR_DELAY > 15), (DEP_DELAY > 15)]
#    - Values:
# tensor([[0, 0],
#         [0, 0],
#         [0, 0],
#         [0, 0],
#         [0, 0],
#         [0, 0]], dtype=torch.int8)

# 4. Valid Sequence Lengths (Metadata):
#    - Description: Effective number of valid flights in the chain before padding
#    - Shape: torch.Size([])
#    - Values: 1

# 5. Raw Delays (Ground Truth):
#    - Shape: torch.Size([6, 2]) (Sequence Length x 2)
#    - Cols: ['ARR_DELAY', 'DEP_DELAY']
#    - Values:
# tensor([[-29,  -5],
#         [  0,   0],
#         [  0,   0],
#         [  0,   0],
#         [  0,   0],
#         [  0,   0]], dtype=torch.int16)
# ------------------------------------------------------------

# 6. PLUS the last tensor